# `triangulate_surface_points()`

The geometry function `nematics3d.triangulate_surface_points()` reconstructs a closed triangular surface mesh from an unordered set of sampled 3D surface points. The returned object is a `PyVista` `PolyData`: its vertices are exactly the input points, while the function infers the triangle connectivity.

The reconstruction uses the point-cloud centroid as a reference center, projects every point radially onto the unit sphere, computes a convex hull there, and transfers the resulting triangle connectivity back to the original coordinates. This makes the method useful for sphere-like, ellipsoidal, and moderately deformed closed surfaces that are approximately star-shaped with respect to their centroid.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy`, `Nematics3D`, and the `PlotPolyData` visualization wrapper used below.


In [ ]:
import numpy as np
import nematics3d as n3d
from nematics3d.classes.visual.plot_polydata import PlotPolyData

## Minimal example: reconstruct a tetrahedron

A tetrahedron is the smallest nondegenerate closed triangular surface in 3D. Passing its four vertices is therefore a compact way to see the basic input-output behavior.


In [ ]:
points = np.array([
    [1.0, 1.0, 1.0],
    [-1.0, -1.0, 1.0],
    [-1.0, 1.0, -1.0],
    [1.0, -1.0, -1.0],
])

mesh = n3d.triangulate_surface_points(points)
print(mesh)
print("same vertices:", np.allclose(mesh.points, points))
print("number of triangles:", mesh.n_cells)

## Inputs and outputs

The public signature is:

```python
triangulate_surface_points(points)
```

`points` must contain at least four 3D points with shape `(N, 3)`. The function returns a `PyVista` `PolyData` whose point coordinates are the original input coordinates. No smoothing, interpolation, or insertion of new vertices is performed. The added information is the triangular connectivity between those vertices.

The function is therefore a surface-reconstruction helper rather than a generic point-cloud fitting routine. It assumes that the input already samples one closed surface sufficiently well.


## Examples


### Reconstruct a deformed sphere

For a more representative example, generate points with approximately uniform angular coverage and deform their radii smoothly. The resulting point cloud still describes a star-shaped closed surface, so radial projection preserves the angular neighborhood structure needed by the triangulation.


In [ ]:
n_points = 300
i = np.arange(n_points, dtype=float)
golden_angle = np.pi * (3.0 - np.sqrt(5.0))
z = 1.0 - 2.0 * (i + 0.5) / n_points
rho = np.sqrt(1.0 - z**2)
phi = golden_angle * i
directions = np.column_stack([rho * np.cos(phi), rho * np.sin(phi), z])

radius = 1.0 + 0.18 * directions[:, 0] * directions[:, 1] + 0.12 * directions[:, 2]**2
surface_points = radius[:, None] * directions
surface_points *= np.array([1.4, 1.0, 0.8])

surface_mesh = n3d.triangulate_surface_points(surface_points)
print("points:", surface_mesh.n_points)
print("triangles:", surface_mesh.n_cells)

### Visualize the triangular mesh with `PlotPolyData`

`triangulate_surface_points()` is responsible only for constructing the mesh. Once the `PolyData` exists, `PlotPolyData` can render that mesh directly without reconstructing its topology. Turning on polygon edges makes the inferred triangles visible.


In [ ]:
surface_plot = PlotPolyData(
    surface_mesh,
    name="triangulated surface",
    color=(0.75, 0.82, 0.95),
    is_show_edges=True,
    edge_color=(0.15, 0.15, 0.15),
    edge_width=1.0,
)

### Wireframe view

For an unobstructed view of the connectivity itself, switch the same visual object to wireframe mode.


In [ ]:
surface_plot.opts.style = "wireframe"
surface_plot.opts.edge_width = 1.5

## Details

Let the input surface samples be $\mathbf{x}_i$ and let their centroid be

$$\mathbf{c}=\frac{1}{N}\sum_i \mathbf{x}_i.$$

Each point is mapped to a direction on the unit sphere,

$$\mathbf{u}_i=\frac{\mathbf{x}_i-\mathbf{c}}{\lVert\mathbf{x}_i-\mathbf{c}\rVert}.$$

`SciPy` then computes the convex hull of the $\mathbf{u}_i$. Because the projected points lie on the sphere, the hull facets provide a triangular connectivity on the sphere. `triangulate_surface_points()` reuses those vertex indices with the original $\mathbf{x}_i$, so the output has the original surface geometry together with the inferred connectivity.

This construction explains why radial distance from the centroid does not directly determine which points are neighbors: connectivity is inferred from their projected directions.


## Assumptions and limitations

The important geometric assumption is that the sampled surface is approximately star-shaped with respect to the point-cloud centroid. Informally, a ray starting at the centroid should encounter the surface only once. Sphere-like and moderately deformed sphere-like surfaces satisfy this naturally; a torus, disconnected surface, or strongly concave surface generally does not.

A separate degenerate case occurs if one sampled surface point is exactly at the centroid. Its projected direction would require division by zero. The function detects this case and raises `ValueError`, but it does not currently choose a replacement center automatically.

Likewise, at least four points are necessary but not sufficient for a valid reconstruction. Degenerate projected point sets can still make the `SciPy` convex-hull calculation fail; such failures are reported as `ValueError`.


## Summary

Use `triangulate_surface_points()` when you have unordered samples of one closed, approximately centroid-star-shaped surface and need a triangular `PolyData` mesh. The function preserves the input vertices and supplies the missing face connectivity. Use `PlotPolyData` when you want to visualize the resulting mesh directly.
